# Problem Sheet 3 — Yang Cheng, yacheng@mpia.de

This sheet mixes TNG data (Problems 1-3) with pure calculator code (Problem 4, reusing the functions from Problem Sheet 2). I don't have TNG access on this machine, so the data-loading cells here are **untested** — I wrote them following the exact same `illustris_python` patterns as sheet 1 (which did run fine on the cluster), but I need to actually run this on the cluster before trusting any number or plot below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py
import illustris_python as il

## 1. Evolution of density contrast on different spatial scales

### 1.1 — matter density vs redshift

I use TNG100-3-Dark throughout Problem 1 (same box as 1.4 needs for the volume comparison with TNG300-3-Dark).

I don't remember the exact snapshot-redshift mapping for TNG100, so instead of guessing snapshot numbers I scan all headers once (cheap, headers only) to build a z(snap) table, then pick the snapshots closest to $z=10,5,2,0$, same `argmin` idiom as sheet 1/2.

In [ ]:
basePath = '/home/tnguser/sims.TNG/TNG100-3-Dark/output'
n_snaps = 100

redshifts = np.array([il.groupcat.loadHeader(basePath, s)['Redshift'] for s in range(n_snaps)])
z_targets = [10, 5, 2, 0]
snaps = [int(np.argmin(np.abs(redshifts - zt))) for zt in z_targets]

for zt, snap in zip(z_targets, snaps):
    print(f'target z={zt:5.1f}  ->  snap {snap:3d}, actual z={redshifts[snap]:.3f}')

### Note on headers: group catalog vs snapshot

`il.groupcat.loadHeader()` reads the **group catalog** header (`groups_NNN.0.hdf5`), whose keys are only:

`BoxSize, FlagDoubleprecision, Git_commit, Git_date, HubbleParam, Ngroups_ThisFile, Ngroups_Total, Nids_ThisFile, Nids_Total, Nsubgroups_ThisFile, Nsubgroups_Total, NumFiles, Omega0, OmegaLambda, Redshift, Time`

`MassTable` and `NumPart_Total` are **not** there, they are in the **snapshot** header (`snapdir_NNN/snap_NNN.0.hdf5`). And `illustris_python` has no `snapshot.loadHeader` (only `groupcat.loadHeader`), so I read the snapshot header with h5py directly, same as in Problem Sheet 1.


In [ ]:
def snap_header(basePath, snap, chunk=0):
    # snapshot header (has MassTable / NumPart_Total, unlike the group catalog header)
    with h5py.File(basePath + '/snapdir_%03d/snap_%03d.%s.hdf5' % (snap, snap, chunk), 'r') as f:
        return dict(f['Header'].attrs)

def dm_particle_mass(basePath, snap):
    # Msun per DM particle
    return snap_header(basePath, snap)['MassTable'][1] * 1e10 / snap_header(basePath, snap)['HubbleParam']

def n_dm_total(basePath, snap):
    # NumPart_Total is uint32, the HighWord carries the overflow for the big boxes
    sh = snap_header(basePath, snap)
    return int(sh['NumPart_Total'][1]) + (int(sh['NumPart_Total_HighWord'][1]) << 32)

# sanity check on the box I use below
sh = snap_header(basePath, 99)
print('snapshot header keys:', sorted(sh.keys()))
print(f"DM particle mass = {dm_particle_mass(basePath, 99):.3e} Msun")
print(f"N_DM total       = {n_dm_total(basePath, 99)}")

### a) column density and 3D density maps at each snapshot

Grid cells a few hundred kpc, as hinted. Particle data is downsampled (keep 1 in `downsample`, boost each kept particle's mass by the same factor to conserve total mass), same recipe the problem sheet suggests.

In [ ]:
downsample = 100      # keep 1/100 of the DM particles
cell_kpc = 300         # 2D column-density grid cell size
cell_3d_Mpc = 1.0       # 3D density grid cell size (coarser, or the grid gets huge)

fig, axes = plt.subplots(1, len(snaps), figsize=(5*len(snaps), 5))

for ax, snap in zip(axes, snaps):
    header = snap_header(basePath, snap)                 # snapshot header, not the group catalog one
    h = header['HubbleParam']
    a = header['Time']
    box = header['BoxSize'] * a / h / 1000              # Mpc
    dm_mass = dm_particle_mass(basePath, snap)           # Msun per real particle
    n_total = n_dm_total(basePath, snap)

    coords = load_dm_coords(basePath, snap, downsample)   # strided read, chunk by chunk (Mpc)
    kept_mass = dm_mass * downsample                       # each kept particle now stands in for N real ones

    # 2D column density map, same recipe as sheet 1's surface density plot
    nbins_2d = max(1, int(box * 1000 / cell_kpc))
    pixel_area = (box / nbins_2d)**2
    weights_2d = np.full(len(coords), kept_mass / pixel_area)
    Sigma, *_ = ax.hist2d(coords[:, 0], coords[:, 1], bins=nbins_2d, range=[[0, box], [0, box]],
        weights=weights_2d, norm=mpl.colors.LogNorm(), cmap='magma')
    ax.set(title=f'z={header["Redshift"]:.1f}', xlabel='x [Mpc]', ylabel='y [Mpc]')
    ax.set_aspect('equal')

    # 3D density on a coarser grid
    nbins_3d = max(1, int(box / cell_3d_Mpc))
    counts_3d, _ = np.histogramdd(coords, bins=nbins_3d, range=[[0, box]]*3)
    cell_vol = (box / nbins_3d)**3
    rho_3d = counts_3d * kept_mass / cell_vol             # Msun/Mpc^3

    rho_avg = n_total * dm_mass / box**3                   # exact box average, no downsampling noise

    print(f'z={header["Redshift"]:.2f}:')
    print(f'  column density range : {Sigma[Sigma>0].min():.3e} - {Sigma.max():.3e} Msun/Mpc^2')
    print(f'  3D density range     : {rho_3d[rho_3d>0].min():.3e} - {rho_3d.max():.3e} Msun/Mpc^3')
    print(f'  3D average density   : {rho_avg:.3e} Msun/Mpc^3')

plt.tight_layout()
plt.show()

### 1.1 small discussion

**[TODO once I can actually run this on the cluster]** — expect the column/3D density ranges to span many orders of magnitude (voids vs. filaments/clusters), and the box average density to stay constant across redshift in comoving terms but I'm computing it in physical Mpc here (box already converted with `a/h/1000`), so the physical average density should grow like $(1+z)^3$ going back in time, same $\rho_m(z)$ scaling from Problem Sheet 2.

### 1.2 — density contrast PDF at z=0, different smoothing scales

$\delta(r,t) = [\rho(r,t)-\bar\rho(t)]/\bar\rho(t)$. I measure it on a 3D grid (cube size = smoothing scale), then look at the PDF of $1+\delta=\rho/\bar\rho$ (safe to put on a log axis since it's always $\geq0$, unlike $\delta$ itself which can be negative).

**Why I don't use `np.histogramdd` here.** The box is 110.7 Mpc, so a 100 kpc cell means $1107^3 = 1.36\times10^9$ cells. A dense grid of that is 10.1 GiB as float64, and `np.histogramdd` internally allocates a `np.bincount` array of $(n+2)^3$ int64 (another 10.2 GiB), reshapes it, slices it and casts it — call it ~30 GiB peak, on top of the ~1 GiB of coordinates. That is what blows up.

But at 100 kpc the grid is also almost entirely **empty**: TNG100-3-Dark has $455^3 = 9.42\times10^7$ DM particles, i.e. 0.069 particles per cell even using every single one. So I never build the grid: I compute the integer cell index of each particle, sort, and run-length-encode to get the counts of the *occupied* cells only. That is bit-identical to `histogramdd` (I checked) and costs ~$N$ instead of ~$n_{bins}^3$. The empty cells are still accounted for — I keep the total cell count and report the empty fraction — they just never need to be stored.

Two further things this fixes:

- **the $-\infty$ crash**: empty cells give $\log_{10}(1+\delta)=-\infty$, and `plt.hist`/`np.histogram` with auto range raises `ValueError: autodetected range ... is not finite`. So the cell would have died at *every* scale, not just 100 kpc. I now pass an explicit bin `range` and histogram only the occupied cells (`density=True` normalizes over what lands in the bins, so this gives exactly the same PDF as the dense version would).
- **the load**: `il.snapshot.loadSubset(...)` pulls the whole $9.42\times10^7\times3$ coordinate array into memory (1.05 GiB float32 / 2.11 GiB float64) and only *then* gets sliced by `[::downsample]`. I read the snapshot chunk by chunk and let HDF5 apply the stride, so the peak is one chunk.

Note on the smoothing scale: the mean interparticle separation in TNG100-3-Dark is $110.7/455 = 0.243$ Mpc, so a 100 kpc cell is *below* the resolution of the simulation. I therefore use every particle (`downsample=1`) at that scale — with `downsample=100` the mean occupancy would be 0.0007 particles/cell and the "PDF" would be pure shot noise. See the discussion below.


In [ ]:
def load_dm_coords(basePath, snap, downsample=1):
    # read PartType1/Coordinates chunk by chunk, letting HDF5 apply the stride,
    # so the full N x 3 array is never in memory at once. Returns physical Mpc.
    sh = snap_header(basePath, snap)
    h, a, nfiles = sh['HubbleParam'], sh['Time'], int(sh['NumFilesPerSnapshot'])
    pieces, offset = [], 0
    for i in range(nfiles):
        with h5py.File(basePath + '/snapdir_%03d/snap_%03d.%d.hdf5' % (snap, snap, i), 'r') as f:
            if 'PartType1' not in f:
                continue
            n = f['PartType1/Coordinates'].shape[0]
            start = (-offset) % downsample      # keep the stride continuous across chunks
            if start < n:
                pieces.append(np.asarray(f['PartType1/Coordinates'][start::downsample],
                                         dtype=np.float32))
            offset += n
    return np.concatenate(pieces) * (a / h / 1000)


def cell_counts(coords, box, nbins):
    # particle counts of the OCCUPIED cells only, without ever allocating the nbins^3 grid.
    # identical to np.histogramdd(...)[0][grid > 0], at O(N) memory instead of O(nbins^3).
    scale = nbins / box
    flat = np.clip((coords[:, 0].astype(np.float64) * scale).astype(np.int64), 0, nbins-1)
    for k in (1, 2):
        flat *= nbins
        flat += np.clip((coords[:, k].astype(np.float64) * scale).astype(np.int64), 0, nbins-1)
    flat.sort()                                  # in place
    new = np.empty(len(flat) + 1, dtype=bool)    # run-length encode the sorted indices
    new[0] = new[-1] = True
    np.not_equal(flat[1:], flat[:-1], out=new[1:-1])
    return np.diff(np.flatnonzero(new))


def delta_grid(basePath, snap, cell_Mpc, downsample=1, verbose=True):
    # returns 1+delta for the occupied cells, plus the empty-cell fraction.
    # the particle mass cancels: 1+delta = count / (N_kept/N_cells)
    sh = snap_header(basePath, snap)
    box = sh['BoxSize'] * sh['Time'] / sh['HubbleParam'] / 1000     # Mpc

    coords = load_dm_coords(basePath, snap, downsample)
    nbins = max(1, int(box / cell_Mpc))
    n_cells = nbins**3
    n_kept = len(coords)

    occ = cell_counts(coords, box, nbins)
    del coords

    mean_per_cell = n_kept / n_cells
    empty_frac = 1 - len(occ) / n_cells
    if verbose:
        print(f'  cell={box/nbins:.3f} Mpc  nbins={nbins}  cells={n_cells:.3e}  '
              f'N_kept={n_kept:.3e}  <N/cell>={mean_per_cell:.3g}  empty={empty_frac:.3%}')
    return occ / mean_per_cell, empty_frac


scales_Mpc = [0.1, 1.0, 10.0]   # 100 kpc, 1 Mpc, 10 Mpc
# 100 kpc is below the mean interparticle separation (0.243 Mpc), so no downsampling there
downsample_per_scale = {0.1: 1, 1.0: 1, 10.0: 10}

fig, ax = plt.subplots(figsize=(7, 6))
for scale in scales_Mpc:
    print(f'{scale} Mpc:')
    one_plus_delta, empty_frac = delta_grid(basePath, 99, scale,
                                            downsample=downsample_per_scale[scale])
    ax.hist(np.log10(one_plus_delta), bins=100, range=(-2, 5), density=True,
            histtype='step', label=f'{scale} Mpc ({empty_frac:.0%} empty cells)')
    print(f'  max log10(1+delta) = {np.log10(one_plus_delta.max()):.2f}')

ax.set(xlabel='log10(1+delta)', ylabel='PDF', yscale='log',
       title='z=0 density contrast PDF (occupied cells)')
ax.legend()
plt.show()

### 1.2 small discussion

**[TODO check on cluster]** Expect strongly asymmetric PDFs: delta is bounded below at -1 (empty cells, which are exactly the ones I count separately rather than store) but essentially unbounded above (dense peaks), so a long tail towards high density. Smaller smoothing scale should give a much wider, more skewed distribution (individual halos/clumps not averaged out), larger scale (10 Mpc) should look closer to a narrow peak around delta~0 since it averages over large, closer-to-homogeneous regions.

**Caveat on the 100 kpc scale.** The mean interparticle separation in TNG100-3-Dark is $110.7/455 = 0.243$ Mpc, so a 100 kpc cube is smaller than the typical spacing between particles even using all of them: the mean occupancy is 0.069 particles per cell and ~93% of cells come out empty. At that scale the PDF is dominated by particle **discreteness** (Poisson shot noise), not by the underlying density field — the high-density tail there is partly "how many particles happened to land in this cell". So the 100 kpc curve should be read as a resolution limit rather than a measurement, and the 1 Mpc and 10 Mpc curves are the trustworthy ones. Worth stating explicitly in the writeup, and worth re-checking against TNG50-4-Dark (box 51.7 Mpc, $270^3$ particles, spacing 0.191 Mpc) which is only marginally better.


### 1.3 — how the PDF changes with redshift

Same `delta_grid` function, fixed smoothing scale (1 Mpc), reusing the `snaps` list from 1.1.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for snap in snaps:
    header = il.groupcat.loadHeader(basePath, snap)     # only Redshift needed here
    print(f"z={header['Redshift']:.1f}:")
    one_plus_delta, empty_frac = delta_grid(basePath, snap, 1.0, downsample=1)
    ax.hist(np.log10(one_plus_delta), bins=100, range=(-2, 5), density=True, histtype='step',
            label=f"z={header['Redshift']:.1f} ({empty_frac:.0%} empty)")
ax.set(xlabel='log10(1+delta)', ylabel='PDF', yscale='log', title='PDF evolution, 1 Mpc smoothing')
ax.legend()
plt.show()

### 1.3 small discussion

**[TODO check on cluster]** Expect the high-density tail to grow longer as the Universe ages (z decreases): at very high z the field should still be close to the near-uniform initial conditions (narrow PDF around delta~0), and structure grows over time via gravitational collapse, building up the rare high-delta tail we already see at z=0 in 1.2.

### 1.4 — TNG100 vs TNG300, same smoothing scale

Same function again, just pointed at a different (bigger) box.

In [ ]:
basePath_300 = '/home/tnguser/sims.TNG/TNG300-3-Dark/output'

fig, ax = plt.subplots(figsize=(7, 6))
for name, bp in [('TNG100-3-Dark', basePath), ('TNG300-3-Dark', basePath_300)]:
    print(f'{name}:')
    one_plus_delta, empty_frac = delta_grid(bp, 99, 1.0, downsample=1)
    ax.hist(np.log10(one_plus_delta), bins=100, range=(-2, 5), density=True,
            histtype='step', label=f'{name} ({empty_frac:.0%} empty)')
    print(f'  max log10(1+delta) = {np.log10(one_plus_delta.max()):.2f}')
ax.set(xlabel='log10(1+delta)', ylabel='PDF', yscale='log', title='z=0, 1 Mpc smoothing')
ax.legend()
plt.show()

### 1.4 small discussion

**[TODO check on cluster]** Expect TNG300 to reach higher maximum density contrast than TNG100 even with identical cell size: rare extreme peaks (massive cluster cores) only show up if the volume is big enough to contain a few of them. TNG100's smaller volume just doesn't sample the rarest, most extreme tail of the distribution.

## 2. Gravitational collapse and the formation of haloes

### 2.1 — cluster overdensity today

No TNG data needed here, this is a pure calculation. Cluster: $M=10^{15}M_\odot$ in $R=2$ Mpc. Mean matter density today from the standard constant $\rho_{crit,0}=2.775\times10^{11}h^2\,M_\odot/\text{Mpc}^3$ (Mpc-unit version of the $1.88\times10^{-29}h^2\,\text{g/cm}^3$ from Problem Sheet 2), times $\Omega_{m,0}$.

In [ ]:
M_cluster, R_cluster = 1e15, 2.0   # Msun, Mpc
V_cluster = 4/3*np.pi*R_cluster**3
rho_cluster = M_cluster / V_cluster

Om0, h_std = 0.3, 0.7
rho_crit0 = 2.775e11 * h_std**2      # Msun/Mpc^3
rho_mean = Om0 * rho_crit0

delta_cluster = rho_cluster/rho_mean - 1
print(f'rho_cluster = {rho_cluster:.3e} Msun/Mpc^3')
print(f'rho_mean    = {rho_mean:.3e} Msun/Mpc^3')
print(f'delta_cluster = {delta_cluster:.1f}, log10(1+delta) = {np.log10(1+delta_cluster):.2f}')

### 2.1 small discussion

delta_cluster comes out to ~730, i.e. log10(1+delta)~2.9 — this is way out past anything the 1 Mpc-smoothed PDFs in Exercise 1 show (those topped out well below this), which makes sense: a cluster core averaged over just its own R=2 Mpc is a far more extreme, rare peak than a random Mpc-sized cell in the box. Ties back to 1.4 too: reaching this kind of overdensity requires either a small enough smoothing aperture centered right on the peak, or (as in 1.4) a large enough volume to even contain such a rare object in the first place.

## 3. The distribution of material within dark-matter haloes

### 3.1 — single halo density profile

Same halo-loading + periodic wrap recipe as sheet 1's Problem 2.1, reused here. Bin radius log-spaced from $0.03R_{200c}$ to $R_{200c}$, as hinted.

In [ ]:
snap = 99
header = il.groupcat.loadHeader(basePath, snap)   # fine here: only BoxSize / Time / HubbleParam
h, a = header['HubbleParam'], header['Time']
box_ckpc = header['BoxSize']                    # ckpc/h, for the periodic wrap
dm_mass = dm_particle_mass(basePath, snap)       # Msun per particle (from the SNAPSHOT header)

halos = il.groupcat.loadHalos(basePath, snap, fields=['Group_M_Crit200', 'Group_R_Crit200', 'GroupPos'])
M200c = halos['Group_M_Crit200'] * 1e10 / h              # Msun
R200c = halos['Group_R_Crit200'] * a / h                  # physical kpc

def load_halo_dm(haloID):
    pos = il.snapshot.loadHalo(basePath, snap, haloID, partType=1, fields=['Coordinates'])
    d = pos - halos['GroupPos'][haloID]
    d = (d + box_ckpc/2) % box_ckpc - box_ckpc/2      # periodic wrap, still in comoving ckpc/h
    return d * a / h                                    # physical kpc

def density_profile(d, r200, n_bins=20):
    r = np.linalg.norm(d, axis=1)
    bins = np.logspace(np.log10(0.03*r200), np.log10(r200), n_bins+1)
    mass, _ = np.histogram(r, bins=bins, weights=np.full(len(r), dm_mass))
    vol = 4/3*np.pi*(bins[1:]**3 - bins[:-1]**3)
    r_mid = np.sqrt(bins[1:]*bins[:-1])
    return r_mid, mass/vol

haloID = int(np.argmin(np.abs(M200c - 1e13)))    # example halo, ~1e13 Msun
d = load_halo_dm(haloID)
r_mid, rho = density_profile(d, R200c[haloID])

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(r_mid, rho, marker='o')
ax.set(xscale='log', yscale='log', xlabel='r [kpc]', ylabel='rho [Msun/kpc^3]',
       title=f'Halo {haloID}, M200c={M200c[haloID]:.2e} Msun')
plt.show()

### 3.1 small discussion

**[TODO check on cluster]** Expect a steeply falling, roughly power-law-ish profile in log-log, steepening at large r — the classic cuspy dark-matter halo shape (NFW-like), much denser in the center than at $R_{200c}$.

### 3.2 — profiles in bins of halo mass

Three mass bins ($10^{11},10^{12},10^{13}\,M_\odot$, $\pm0.2$ dex), average profile per bin (capped to 40 haloes per bin just for speed). Then repeat with $r$ normalized by $R_{200c}$.

In [ ]:
mass_targets = [1e11, 1e12, 1e13]
dex = 0.2
n_max = 40   # cap per bin, just for speed

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
profiles_norm = {}

for ax, Mt in zip(axes, mass_targets):
    sel = np.where((M200c > Mt*10**-dex) & (M200c < Mt*10**dex))[0][:n_max]
    stack, stack_norm = [], []
    for hid in sel:
        d = load_halo_dm(hid)
        r_mid, rho = density_profile(d, R200c[hid])
        stack.append(rho)
        stack_norm.append(rho)   # same rho, just plotted vs r/R200c below
    avg_rho = np.nanmean(stack, axis=0)
    profiles_norm[Mt] = (r_mid/R200c[sel[0]], avg_rho)   # approx common x-grid, R200c similar within the bin
    ax.plot(r_mid, avg_rho)
    ax.set(xscale='log', yscale='log', title=f'M~{Mt:.0e} Msun (N={len(sel)})', xlabel='r [kpc]')
    print(f'M~{Mt:.0e}: central density (innermost bin) = {avg_rho[0]:.3e} Msun/kpc^3')

axes[0].set_ylabel('rho [Msun/kpc^3]')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
for Mt, (x, y) in profiles_norm.items():
    ax.plot(x, y, label=f'M~{Mt:.0e}')
ax.set(xscale='log', yscale='log', xlabel='r / R200c', ylabel='rho [Msun/kpc^3]')
ax.legend()
plt.show()

### 3.2 small discussion

**[TODO check on cluster]** Expect more massive haloes to have higher central densities and overall normalization (more mass packed in), but a roughly similar *shape* once radius is normalized by $R_{200c}$ — that self-similarity (or lack of it) is exactly what the normalized plot is meant to show.

### 3.3 — fitting NFW profiles, scale radius vs halo mass

$\rho_{NFW}(r) = \rho_s / [(r/r_s)(1+r/r_s)^2]$. Fit $\rho_s, r_s$ per halo with `scipy.optimize.curve_fit`, using a sample of haloes across a range of masses.

In [ ]:
from scipy.optimize import curve_fit

def nfw(r, rho_s, r_s):
    x = r/r_s
    return rho_s / (x*(1+x)**2)

sample = np.where(M200c > 1e11)[0]
rng = np.random.default_rng(0)
sample = rng.choice(sample, size=min(200, len(sample)), replace=False)

r_s_fit, M_fit = [], []
for hid in sample:
    d = load_halo_dm(hid)
    r_mid, rho = density_profile(d, R200c[hid])
    good = rho > 0
    try:
        popt, _ = curve_fit(nfw, r_mid[good], rho[good], p0=[rho[good][0], R200c[hid]/5], maxfev=5000)
        if popt[1] > 0:
            r_s_fit.append(popt[1])
            M_fit.append(M200c[hid])
    except RuntimeError:
        continue   # fit didn't converge for this halo, just skip it

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(M_fit, r_s_fit, s=10, alpha=0.6)
ax.set(xscale='log', yscale='log', xlabel='M200c [Msun]', ylabel='r_s [kpc]')
plt.show()

### 3.3 small discussion

**[TODO check on cluster]** Expect scale radius to increase with halo mass (roughly a power law), since more massive haloes are both denser and physically bigger overall — same trend as $R_{200c}$ growing with mass, just for the inner NFW scale instead.

## 4. Build your own Python Cosmological Calculator (Part II)

### Growth factor $D_+(z)$, Carroll+1992 fitting formula

This part doesn't need TNG at all, it's pure calculator code like Problem Sheet 2, so I can actually test it here.

Carroll, Press & Turner (1992) fitting formula for the growth suppression factor $g(\Omega_m,\Omega_\Lambda)$:

$g(\Omega_m,\Omega_\Lambda) = \dfrac{5}{2}\Omega_m \Big/ \left[\Omega_m^{4/7} - \Omega_\Lambda + \left(1+\dfrac{\Omega_m}{2}\right)\left(1+\dfrac{\Omega_\Lambda}{70}\right)\right]$

evaluated with $\Omega_m(z),\Omega_\Lambda(z)$ (not today's values!), and normalized so $D_+(0)=1$:

$D_+(z) = \dfrac{1}{1+z}\cdot\dfrac{g(\Omega_m(z),\Omega_\Lambda(z))}{g(\Omega_{m,0},\Omega_{\Lambda,0})}$

I copy over the minimal $E(z)$/$\Omega_i(z)$ machinery from Problem Sheet 2's 1.3 (notebooks don't share state, so I can't just import the function directly).

In [ ]:
def E_gen(z, Om, Or, OL):
    Ok = 1 - Om - Or - OL
    return np.sqrt(Or*(1+z)**4 + Om*(1+z)**3 + Ok*(1+z)**2 + OL)

def Omega_i_gen(z, Oi0, n, Om, Or, OL):
    return Oi0*(1+z)**n / E_gen(z, Om, Or, OL)**2

def g_carroll(Om, OL):
    return 2.5*Om / (Om**(4/7) - OL + (1+Om/2)*(1+OL/70))

def D_plus(z, Om0, Or0, OL0):
    Omz = Omega_i_gen(z, Om0, 3, Om0, Or0, OL0)
    OLz = Omega_i_gen(z, OL0, 0, Om0, Or0, OL0)
    return (g_carroll(Omz, OLz) / g_carroll(Om0, OL0)) / (1+z)

# quick sanity check: for EdS this should reduce exactly to D+(z) = 1/(1+z)
print('EdS D+(z=1) =', D_plus(1, 1, 0, 0), ' expect 0.5')

In [ ]:
z = np.linspace(0, 10, 200)
scenarios4 = {
    'EdS (Om=1)':                dict(Om0=1.0, Or0=0.0, OL0=0.0),
    'Low-density (Om=0.3,OL=0)': dict(Om0=0.3, Or0=0.0, OL0=0.0),
    'Standard (Om=0.3,OL=0.7)':  dict(Om0=0.3, Or0=0.0, OL0=0.7),
}

fig, ax = plt.subplots(figsize=(7, 6))
for name, p in scenarios4.items():
    ax.plot(z, [D_plus(zi, **p) for zi in z], label=name)
ax.set(xlabel='z', ylabel='D+(z)', yscale='log', title='Linear growth factor')
ax.legend()
plt.show()

### 4 small discussion

EdS grows exactly as $D_+\propto a\propto(1+z)^{-1}$ (confirmed by the sanity check above). Low-density and Standard both start at $D_+(0)=1$ by construction, but decline *less steeply* than EdS going back in z, with Low-density staying highest of the three at any given z, Standard (LCDM) in between. I checked the actual numbers up to z=1000: they never merge onto one curve, they approach constant *ratios* relative to EdS ($\sim2.2$ for Low-density, $\sim1.29$ for Standard) rather than converging to the same value — makes sense since each curve is normalized by its own $g(\Omega_{m,0},\Omega_{\Lambda,0})$ today, and that normalization constant is different for each model and never goes away. So on this log-log plot all three eventually become straight, parallel lines with the same slope ($D_+\propto(1+z)^{-1}$, since $\Omega_m(z)\to1$ for all of them early on, same point as Problem Sheet 2), just offset from each other.